# Persian Emotional TTS
## نوت‌بوک مرحله‌ای پروژه در Google Colab

این نوت‌بوک برای تولید گفتار فارسی با کنترل لحن و احساس، بر اساس نقشه راه پنج‌فازی پروژه تنظیم شده است.

- **محیط اجرا:** Google Colab
- **ذخیره‌سازی پایدار:** Google Drive
- **مسیر پروژه:** `/content/drive/MyDrive/Persian_Emotional_TTS`
- **وضعیت فعلی:** گام‌های ۱.۱ و ۱.۲ اجرا شده‌اند؛ گام ۱.۳ برای دریافت و اعتبارسنجی پیکره آماده است.

> سلول‌ها را به ترتیب و فعلاً فقط تا پایان گام ۱.۳ اجرا کن. سایر گام‌ها فعلاً فقط ساختار و محل مشخص دارند و انجام‌شده محسوب نمی‌شوند.

## نقشه راه پروژه

### فاز ۱: زیرساخت، ابزارها و آماده‌سازی داده
- [x] **۱.۱:** راه‌اندازی Colab، اتصال Drive و ساختار پوشه‌بندی
- [x] **۱.۲:** نصب کتابخانه‌های کلیدی AI، پردازش صوت و مدل‌سازی
- [ ] **۱.۳:** جمع‌آوری پیکره داده‌های داستانی و صوتی
- [ ] **۱.۴:** پیش‌پردازش متن فارسی (G2P) و Force Alignment
- [ ] **۱.۵:** برچسب‌گذاری احساسی اولیه (Pseudo-Labeling)

### فاز ۲: ماژول درک متن و تحلیل احساس
- [ ] **۲.۱:** انتخاب مدل زبانی سبک فارسی
- [ ] **۲.۲:** استخراج بردارهای Emotion/Prosody از متن

### فاز ۳: مدل آکوستیک و ووکودر
- [ ] **۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS
- [ ] **۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی احساس
- [ ] **۳.۳:** اتصال BigVGAN برای تولید صوت

### فاز ۴: اجرای بلادرنگ و استریم
- [ ] **۴.۱:** Sentence/Clause Chunking
- [ ] **۴.۲:** Async Pipeline برای کاهش تأخیر
- [ ] **۴.۳:** ONNX و WebSocket Audio Streaming

### فاز ۵: ارزیابی و بهبود
- [ ] **۵.۱:** ارزیابی انسانی و MOS
- [ ] **۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال

## قبل از اجرا

1. نوت‌بوک را در [Google Colab](https://colab.research.google.com/) باز کن.
2. از مسیر **Runtime → Change runtime type → Hardware accelerator** گزینه **GPU** را انتخاب کن.
3. سلول‌ها را از بالا به پایین اجرا کن.
4. هنگام اتصال Google Drive، دسترسی لازم را تأیید کن.
5. پس از قراردادن داده و اجرای گام ۱.۳، خروجی سلول پایانی همان گام را برای بررسی نگه دار.

# فاز ۱ — زیرساخت، ابزارها و آماده‌سازی داده
## گام ۱.۱ — بررسی محیط، اتصال Drive و ساختار پروژه

### سلول ۱: بررسی Python، PyTorch و GPU

این سلول نصب جدیدی انجام نمی‌دهد و فقط وضعیت runtime فعلی Colab را گزارش می‌کند.

In [ ]:
import platform
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA runtime:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: Select a GPU from Runtime > Change runtime type.")
except ImportError:
    print("PyTorch is not installed; it will be handled in step 1.2.")

### سلول ۲: اتصال Google Drive

در اولین اجرا، Colab پنجره اجازه دسترسی به Drive را نمایش می‌دهد. حساب موردنظر را انتخاب و دسترسی را تأیید کن.

In [ ]:
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

assert Path("/content/drive/MyDrive").is_dir(), "Google Drive mount failed."
print("Google Drive is ready.")

### سلول ۳: ساخت پوشه‌ها و فایل پیکربندی

اجرای این سلول امن و قابل تکرار است؛ فایل‌های صوتی یا مدل‌های موجود حذف نمی‌شوند.

In [ ]:
import json
from pathlib import Path

PROJECT_NAME = "Persian_Emotional_TTS"
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
DIRECTORIES = (
    "data/raw",
    "data/processed",
    "models/acoustic",
    "models/emotion_extractor",
    "models/vocoder",
    "configs",
)

for relative_path in DIRECTORIES:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)
    print(f"Ready: {relative_path}/")

project_config = {
    "project_name": PROJECT_NAME,
    "project_root": str(PROJECT_ROOT),
    "directories": list(DIRECTORIES),
}
config_path = PROJECT_ROOT / "configs" / "project_config.json"
config_path.write_text(
    json.dumps(project_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


### سلول ۴: اعتبارسنجی ساختار پروژه

این سلول وجود همه پوشه‌ها و فایل پیکربندی را بررسی می‌کند.

In [ ]:
missing_directories = [
    relative_path
    for relative_path in DIRECTORIES
    if not (PROJECT_ROOT / relative_path).is_dir()
]
assert not missing_directories, f"Missing directories: {missing_directories}"
assert config_path.is_file(), "Project configuration was not created."

print("Project root:", PROJECT_ROOT)
print("Structure verification: PASSED")
print("Step 1.1 setup: VERIFIED")

## گام ۱.۲ — نصب کتابخانه‌های کلیدی

### سلول ۵: نصب یا تکمیل کتابخانه‌ها

Colab معمولاً PyTorch متناسب با CUDA خودش را از قبل دارد؛ بنابراین نسخه CUDA آن را بی‌دلیل تغییر نمی‌دهیم. اگر PyTorch/Torchaudio/Torchvision موجود نباشند، این سلول آن‌ها را هم نصب می‌کند.

کتابخانه‌های اختصاصی Matcha-TTS و BigVGAN عمداً در این مرحله نصب نمی‌شوند تا در فاز سوم، نسخه‌های سازگار با معماری پروژه انتخاب شوند.

In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
]

for package in ("torch", "torchaudio", "torchvision"):
    if importlib.util.find_spec(package) is None:
        packages.append(package)

print("Installing/checking:", ", ".join(packages))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)
print("Package installation completed.")

### سلول ۶: تست import و نسخه‌های کتابخانه‌ها

اجرای این سلول تایید می‌کند که همه کتابخانه‌های موردنیاز گام ۱.۲ بدون خطا import می‌شوند و در صورت وجود GPU، یک تست ساده CUDA نیز انجام می‌شود.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version

packages_to_verify = (
    "torch",
    "torchaudio",
    "torchvision",
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
)

failed_imports = []
for package in packages_to_verify:
    try:
        importlib.import_module(package)
        try:
            package_version = version(package)
        except PackageNotFoundError:
            package_version = "version unavailable"
        print(f"OK  {package:<14} {package_version}")
    except Exception as exc:
        failed_imports.append(package)
        print(f"FAIL {package}: {type(exc).__name__}: {exc}")

if failed_imports:
    raise RuntimeError(
        "Step 1.2 is not complete. Failed imports: " + ", ".join(failed_imports)
    )

import torch

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    test_tensor = torch.rand(1024, device="cuda")
    print("CUDA tensor test: PASSED; mean =", round(test_tensor.mean().item(), 4))
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Imports passed, but no GPU is active in this runtime.")

print("\nEnvironment verification: PASSED")

## گام ۱.۲ — اجرا و تأیید شد

کتابخانه‌های موردنیاز نصب و import آن‌ها بررسی شده است. اکنون سلول‌های گام ۱.۳ را ادامه بده. داده‌های پروژه همچنان روی Google Drive باقی می‌مانند.

## گام ۱.۳ — جمع‌آوری و ثبت پیکره داستانی و صوتی

در این گام، هر فایل صوتی باید متن داستانی دقیق خودش را داشته باشد. نوت‌بوک فایل‌ها را جفت می‌کند، اطلاعات صوت را می‌خواند و یک manifest استاندارد می‌سازد.

**قالب ورودی:**

- صوت: WAV، FLAC، OGG یا MP3؛ ترجیحاً WAV
- متن: فایل UTF-8 با پسوند `.txt`
- نام و مسیر نسبی فایل صوت و متن باید یکسان باشد؛ فقط پسوند متفاوت است.
- از داده‌هایی استفاده کن که اجازه استفاده از آن‌ها را داری.

مثال یک جفت معتبر:

```text
audio/speaker_01/story_001.wav
transcripts/speaker_01/story_001.txt
```

برچسب احساس در این گام لازم نیست؛ Pseudo-Labeling در گام ۱.۵ انجام می‌شود.

### سلول ۷: آماده‌سازی محل دریافت پیکره

این سلول زیرپوشه‌های `audio` و `transcripts` را داخل `data/raw` می‌سازد و یک راهنمای کوچک روی Drive ذخیره می‌کند. هیچ فایل داده‌ای حذف نمی‌شود.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
assert Path("/content/drive/MyDrive").is_dir(), "Run the Google Drive mounting cell first."
assert PROJECT_ROOT.is_dir(), "Project root not found; run the step 1.1 setup cells first."

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
AUDIO_DIR = RAW_DATA_DIR / "audio"
TRANSCRIPT_DIR = RAW_DATA_DIR / "transcripts"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

collection_guide = """Persian Emotional TTS - Raw corpus guide

Place speech files under audio/ and their matching transcripts under transcripts/.
Each audio file must have a UTF-8 .txt file with the same relative path and name.

Example:
  audio/speaker_01/story_001.wav
  transcripts/speaker_01/story_001.txt

Keep the original files unchanged. Text normalization and audio preprocessing belong to step 1.4.
Emotion labels are not required at this stage.
"""
(RAW_DATA_DIR / "README.txt").write_text(collection_guide, encoding="utf-8")

print("Audio inbox:", AUDIO_DIR)
print("Transcript inbox:", TRANSCRIPT_DIR)
print("Raw corpus folders are ready.")

### اقدام لازم در Google Drive

قبل از اجرای سلول بعد، داده واقعی را در Drive قرار بده:

```text
Persian_Emotional_TTS/
└── data/
    └── raw/
        ├── audio/
        │   └── speaker_01/
        │       ├── story_001.wav
        │       └── story_002.wav
        └── transcripts/
            └── speaker_01/
                ├── story_001.txt
                └── story_002.txt
```

هر فایل `.txt` باید فقط متن همان فایل صوتی را داشته باشد. می‌توانی چند گوینده با پوشه‌های جدا داشته باشی. برای تست مسیر، چند جفت نمونه کافی است؛ کامل‌بودن پیکره نهایی را پس از بررسی تعداد و تنوع داده مشخص می‌کنیم.

### سلول ۸: جفت‌سازی صوت و متن، کنترل فایل‌ها و ساخت metadata

این سلول هیچ صوتی را تغییر نمی‌دهد. اگر فایل بدون متن، متن بدون صوت، نام تکراری یا صوت خراب وجود داشته باشد، با پیام خطا متوقف می‌شود تا ابتدا داده اصلاح شود.

In [ ]:
import csv
from collections import Counter
from pathlib import Path

import soundfile as sf

AUDIO_EXTENSIONS = {".wav", ".flac", ".ogg", ".mp3"}


def index_files_by_relative_stem(root, extensions):
    """Index files by relative path without suffix and reject ambiguous names."""
    indexed = {}
    for file_path in sorted(root.rglob("*")):
        if not file_path.is_file() or file_path.suffix.lower() not in extensions:
            continue
        key = file_path.relative_to(root).with_suffix("")
        if key in indexed:
            raise ValueError(
                f"Duplicate relative name for {key}: {indexed[key]} and {file_path}"
            )
        indexed[key] = file_path
    return indexed


audio_files = index_files_by_relative_stem(AUDIO_DIR, AUDIO_EXTENSIONS)
transcript_files = index_files_by_relative_stem(TRANSCRIPT_DIR, {".txt"})

if not audio_files:
    raise FileNotFoundError(
        "No audio files found. Add matching audio/transcript pairs under "
        f"{AUDIO_DIR} and {TRANSCRIPT_DIR}, then rerun this cell."
    )

missing_transcripts = sorted(set(audio_files) - set(transcript_files))
unused_transcripts = sorted(set(transcript_files) - set(audio_files))
if missing_transcripts or unused_transcripts:
    raise ValueError(
        "Audio/transcript mismatch. "
        f"Missing transcripts: {missing_transcripts}; "
        f"Transcripts without audio: {unused_transcripts}"
    )

rows = []
file_errors = []
for key in sorted(audio_files):
    audio_path = audio_files[key]
    transcript_path = transcript_files[key]
    try:
        info = sf.info(str(audio_path))
        if info.samplerate <= 0 or info.frames <= 0:
            raise ValueError("Audio has no usable frames or sample rate.")
        text = transcript_path.read_text(encoding="utf-8").strip()
        if not text:
            raise ValueError("Transcript is empty.")
    except Exception as exc:
        file_errors.append(f"{key}: {type(exc).__name__}: {exc}")
        continue

    speaker_id = key.parts[0] if len(key.parts) > 1 else "unknown"
    rows.append({
        "utterance_id": "__".join(key.parts),
        "audio_path": audio_path.relative_to(PROJECT_ROOT).as_posix(),
        "text": text,
        "speaker_id": speaker_id,
        "duration_seconds": round(info.frames / info.samplerate, 3),
        "sample_rate": info.samplerate,
        "channels": info.channels,
    })

if file_errors:
    raise ValueError("Invalid corpus files:\n" + "\n".join(file_errors))

metadata_path = RAW_DATA_DIR / "metadata.csv"
fieldnames = (
    "utterance_id",
    "audio_path",
    "text",
    "speaker_id",
    "duration_seconds",
    "sample_rate",
    "channels",
)
with metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)

total_seconds = sum(row["duration_seconds"] for row in rows)
sample_rates = Counter(row["sample_rate"] for row in rows)
speakers = {row["speaker_id"] for row in rows}
short_clips = sum(row["duration_seconds"] < 1 for row in rows)
long_clips = sum(row["duration_seconds"] > 30 for row in rows)

print("Validated utterances:", len(rows))
print("Speakers:", len(speakers))
print("Total duration (minutes):", round(total_seconds / 60, 2))
print("Sample rates:", dict(sample_rates))
print("Clips shorter than 1 second:", short_clips)
print("Clips longer than 30 seconds:", long_clips)
print("Metadata:", metadata_path)
print("Corpus file validation: PASSED")

### سلول ۹: بازخوانی مستقل manifest و کنترل نهایی ورودی

این سلول از فایل CSV ساخته‌شده استفاده می‌کند تا مطمئن شویم manifest قابل خواندن است و همه مسیرهای صوتی و متن‌های آن معتبرند. نتیجه این سلول را برای بررسی کامل‌بودن پیکره ارسال کن.

In [ ]:
import csv

assert metadata_path.is_file(), "metadata.csv was not created; rerun the previous cell."
with metadata_path.open("r", encoding="utf-8", newline="") as csv_file:
    manifest_rows = list(csv.DictReader(csv_file))

assert manifest_rows, "The corpus manifest is empty."
for row in manifest_rows:
    assert (PROJECT_ROOT / row["audio_path"]).is_file(), row["audio_path"]
    assert row["text"].strip(), f"Empty text for {row['utterance_id']}"
    assert float(row["duration_seconds"]) > 0, row["utterance_id"]

print("Manifest rows:", len(manifest_rows))
print("Manifest and source file verification: PASSED")
print("Next: review corpus size, speaker coverage, and transcript quality.")

## گام‌های بعدی فاز ۱ — هنوز پیاده‌سازی نشده‌اند

### گام ۱.۴: پیش‌پردازش متن و همگام‌سازی
زیربخش‌ها: نرمال‌سازی متن فارسی، G2P، Force Alignment و ساخت فایل‌های alignment.

### گام ۱.۵: Pseudo-Labeling احساسی
زیربخش‌ها: انتخاب دسته‌های احساس، تولید برچسب اولیه و کنترل کیفیت برچسب‌ها.

# فاز ۲ — Context & Emotion Extractor
## وضعیت: در انتظار تکمیل فاز ۱

- **گام ۲.۱:** انتخاب مدل زبانی سبک فارسی، مانند ParsBERT یا نسخه کوچک Qwen.
- **گام ۲.۲:** Fine-tuning برای استخراج Emotion/Prosody Vectors از متن داستان.

# فاز ۳ — Acoustic Model & Vocoder
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS.
- **گام ۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی بردار احساس.
- **گام ۳.۳:** اتصال BigVGAN برای تولید صوت باکیفیت و حفظ جزئیات عاطفی.

# فاز ۴ — Real-Time Pipeline
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۴.۱:** Sentence/Clause Chunking هوشمند.
- **گام ۴.۲:** طراحی Async Pipeline برای کاهش تأخیر اولیه.
- **گام ۴.۳:** صادرات ONNX و استریم صوت با WebSocket.

# فاز ۵ — Evaluation & Fine-Tuning
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۵.۱:** ارزیابی ذهنی و انسانی کیفیت لحن‌ها با MOS.
- **گام ۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال‌های احتمالی.